# 10. 최종 판단모형 선발 — 동일조건 비교실험이 노트북은 계산을 하지 않는다. 계산은 `src/model/model_selection.py` 와`src/run_model_selection.py` 에 있고, 여기서는 `outputs/model_selection/` 의 실행결과와 근거만 표시한다.재실행:```bashpython src/run_model_selection.py --replicates 100python -m pytest tests/test_model_selection.py -q````outputs/independent_audit/` 의 기존 감사 결과는 읽기만 하며 수정하지 않는다.## 비교 대상| 모형 | 구조 | 쓰는 선호정보 ||---|---|---|| MRSORT_box_only | 현행 ELECTRE TRI-B(q=p=0), 허용 파라미터 박스만 | 없음 || MRSORT_rc / MRSORT_rc_vrc | 같은 구조 + 참조사례(RC) / + 가상사례(VRC) | 연구진 규범 || COUNT-CORE | 가중치·lambda를 고르지 않고 통과 기준 개수만 사용 | 없음 || PARETO_boundary | 경계 프로파일에 대한 지배관계 | 없음 || ROR (UTADIS-GMS) | 가법 비감소 가치함수 + 계급 문턱의 필연·가능 배정 | 층별 || PROMETHEE-II + SMAA | 같은 w 박스에서의 순위수용도 | 없음(서술용) |모든 집합형 출력은 "명시한 전제 아래 가능한 단계"이며 확률이 아니다. 현장 정답이 없으므로 정확도는 계산하지 않는다.

In [ ]:
import jsonfrom pathlib import Pathimport pandas as pdpd.set_option('display.width', 200)pd.set_option('display.max_columns', 60)OUT = Path('..') / 'outputs' / 'model_selection'if not OUT.exists():    OUT = Path('outputs') / 'model_selection'def table(name):    return pd.read_csv(OUT / (name + '.csv'))summary = json.loads((OUT / 'summary.json').read_text(encoding='utf-8'))summary

## 1. 공통 입력모든 후보가 같은 180행·완전관측 160행·최신분기 10개 업종을 쓴다.g4는 감사에서 수정한 "비교 가능한 과거 고용 이력" 기준이다.

In [ ]:
common = table('common_input')common[common.quarter == summary['latest_quarter']]

## 2. 가중치를 고르지 않으면 무엇이 결정되는가 (COUNT-CORE)허용 박스(각 w = .10~.40, 합 1, lambda = .50~.75)에서 경계 통과 기준 개수 k 만으로concordance 의 최소·최대가 정해진다. k <= 1 이면 어떤 파라미터에서도 통과할 수 없고,k = 4 이면 어떤 파라미터에서도 통과한다. 규범 의존은 전부 k = 2, 3 구간에 있다.

In [ ]:
table('count_core_bounds')

In [ ]:
core = table('count_core')core['weight_free_verdict'].value_counts()

## 3. 참조제약 층별 가능 단계 집합 (MRSort, 연속공간 정확해)q=p=0 으로 고정하면 concordance 가 w 에 선형이므로 (w, lambda) 연속공간 전체를 LP로 정확히 푼다.유한 표본이 놓치는 배정이 없다.

In [ ]:
table('mrsort_layers').head(20)

## 4. UTADIS-GMS / Robust Ordinal Regression가법 가치함수 U(x) = sum_j u_j(g_j) 와 계급 문턱 t1 <= t2 의 필연·가능 배정.선호정보가 없으면 어떤 배정도 배제되지 않는다(3단계 전부 가능). 즉 ROR 은 선호정보 없이는 작동하지 않는다.

In [ ]:
table('ror_preference_consistency')

In [ ]:
table('ror_layers').head(20)

### 가법 모형이 고용증거 게이트를 표현하지 못하는 행"고용감소 증거가 없으면 생산이 크게 감소해도 관찰"이라는 규범은 비보상적(non-compensatory) 규칙이다.가법 모형은 이를 가상사례 VRC5 한 건으로만 배우며, g3 가 VRC5 프로파일보다 큰 행에서는 여전히 추가확인 이상을 허용한다.

In [ ]:
table('gate_representation_check')

## 5. Pareto / 부분순서지배관계는 가중치를 쓰지 않는다. 대신 대부분의 쌍이 비교불가로 남는다.

In [ ]:
display(table('pareto_partial_order_stats'))display(table('pareto_partial_order').query("scope == 'latest_quarter'"))

### 기준 중복 진단g1·g2·g4 는 같은 고용계열에서 파생된다. 각 기준을 뺐을 때 새로 생기는 지배 쌍 수로어느 기준이 실제로 지배관계를 막고 있는지 본다.

In [ ]:
table('pareto_criterion_decisiveness')

## 6. 동일조건 비교### 판별력과 판단 불확실성

In [ ]:
table('comparison_discriminating_power')

### 규범 의존성 — 선호정보를 넣었을 때 바뀌는 행 수

In [ ]:
table('comparison_normative_dependence')

### 필연 배정으로 3단계가 실제로 지지되는가

In [ ]:
table('q6_stage_support')

### 개정 강건성주의: 가능 단계 집합이 넓은 모형은 Jaccard 가 자동으로 높아진다.따라서 이 표의 수치만으로 "가장 강건한 모형"을 고를 수 없다. 판별력 표와 함께 읽어야 한다.

In [ ]:
table('comparison_revision_robustness')

## 7. 최신분기 실제 행동 차이

In [ ]:
table('latest_quarter_comparison')

In [ ]:
table('q1_latest_convergence')

### 왜 그렇게 배정되는가 — 기준별 경계 통과 근거

In [ ]:
table('latest_quarter_reasons')

## 8. 차이의 귀속 — 자료인가 정책선호인가

In [ ]:
attr = table('q23_difference_attribution')display(attr[['changed_by_preference_information',              'changed_by_aggregation_form_same_preference',              'changed_by_aggregation_form_no_preference']].sum())display(attr.groupby('industry')[['changed_by_preference_information',                                  'changed_by_aggregation_form_same_preference']].sum())

## 9. MRSort 가 Pareto 보다 추가로 만드는 판단

In [ ]:
table('q4_added_value_over_pareto').head(20)

## 10. SMAA-PROMETHEE (서술용, 채택하지 않음)완전순위를 강제하므로 판별 근거가 없는 업종에도 순위를 부여한다.순위수용도는 표집분포에 조건부인 빈도이며 위기 확률이 아니다.

In [ ]:
table('promethee_smaa_latest')

## 11. 시간축 보조분석 — 변화점 탐지분기 34개의 단일 평균이동 순열검정. 다중검정 보정을 하지 않았고, 유의한 업종은 없다.현재 표본에서는 g4 가 제공하지 못하는 새로운 판단정보가 추가되지 않는다.

In [ ]:
table('changepoint_employment')